# Leakage-safe velocity prediction from cleaned IMU data

This notebook connects the deterministic IDR sensor pipeline to a fair velocity-model comparison. It displays original and cleaned measurements, audits preprocessing losses, simulates GNSS blackouts, performs journey-grouped cross-validation, and evaluates errors in real speed units.

The key correction is the target formulation. A short IMU window describes changes in motion, but steady travel at 30 km/h can look like steady travel at 90 km/h. The primary models therefore start with the last trustworthy GNSS speed and learn the change in speed since that anchor. Cleaned forward-acceleration integration remains an input feature and a deterministic baseline, rather than a base estimate the model must undo.

## Problems being addressed

- Direct speed models regress toward the dataset mean because absolute speed is not observable from steady-motion IMU alone.
- Randomly splitting overlapping windows would leak near-duplicates into validation. Every split here keeps complete journeys together.
- A few large journeys dominate the usable data, while earlier preprocessing did not attribute every rejected row.
- The old RF used only 24 range statistics and missed vibration shape and frequency.
- Twenty-five neural epochs with patience five can stop too early. The full profile allows 200 epochs, patience 20, and restores the best fold checkpoint.
- One overall MAE hides high-speed and long-outage failures, so results are sliced by journey, speed, and blackout horizon.

In [ ]:
from __future__ import annotations
import json, sys, time
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'idr_backend').is_dir():
            return candidate
    raise RuntimeError('Open this notebook inside the dead reckoning repository.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_DIR = REPO_ROOT / 'notebooks'
for local_path in (REPO_ROOT / 'src', NOTEBOOK_DIR):
    if str(local_path) not in sys.path:
        sys.path.insert(0, str(local_path))
RAW_DATA_DIR = REPO_ROOT / 'SIH-2-main' / 'SIH-2-main' / 'IOVNBD-Speed-Prediction' / 'data' / 'raw'
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'anchored_velocity_comparison'
assert RAW_DATA_DIR.is_dir(), f'Missing data: {RAW_DATA_DIR}'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(REPO_ROOT, RAW_DATA_DIR, ARTIFACT_DIR, sep='\n')

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from velocity_experiment import (
    CONTEXT_FEATURE_NAMES, RAW_FEATURE_NAMES, VELOCITY_MODEL_FEATURE_NAMES,
    ExperimentConfig, assert_group_isolation, build_blackout_dataset,
    choose_grouped_holdout, config_dict, detailed_error_frame,
    fit_final_absolute_neural, fit_final_forest, fit_final_neural,
    fold_audit_frame, make_grouped_cv_folds, measure_latency_ms,
    neural_parameter_samples, paired_paths, predict_final_absolute_neural,
    predict_final_forest, predict_final_neural, read_aligned_journey,
    preprocessing_physics_audit, regression_metrics, replay_clean_journey, rf_imu_features, rf_parameter_grid,
    transform_context, transform_sequence, tune_absolute_neural_baseline,
    tune_neural_family, tune_random_forest,
)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python {sys.version.split()[0]} | PyTorch {torch.__version__} | {DEVICE}')
if DEVICE.type == 'cuda': print(torch.cuda.get_device_name(0))

## Configuration

Full runs 20 reproducibly sampled candidates for each neural family, the complete 108-point RF grid, five grouped folds, and early stopping up to 200 epochs. Smoke is only a quick wiring check and must not be reported as a model result. A full search can take several hours on a GTX 1650.

In [ ]:
SEARCH_PROFILE = 'full'  # Complete 20 GRU + 20 CNN + 108 RF grouped search.
ALLOW_DIAGNOSTIC_FULL_RUN = True
REBUILD_PREPROCESSING = False
CONFIG = ExperimentConfig(search_profile=SEARCH_PROFILE)
print(json.dumps(config_dict(CONFIG), indent=2))
print('GRU candidates:', len(neural_parameter_samples('gru', CONFIG)))
print('CNN candidates:', len(neural_parameter_samples('cnn', CONFIG)))
print('RF candidates: ', len(rf_parameter_grid(CONFIG)))

## 1. Original phone and CAN data

S files provide phone accelerometer, gyroscope, and GPS speed. V files provide the CAN-bus target. They are aligned by elapsed time within 75 ms, never by blindly truncating rows. The source GPS header says Kmh, but the recorded values numerically behave as m/s: multiplying them by 3.6 aligns them with the independent CAN km/h traces. The ingestion helper documents and applies this dataset-specific correction. GPS is used only for past-only calibration evidence and the blackout-start anchor; CAN speed is a label only.

In [ ]:
pairs = paired_paths(RAW_DATA_DIR)
example_frame = read_aligned_journey(*pairs[0], CONFIG)
original_columns = ['journey_id', 'timestamp', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'gps_speed_file_value', 'gps_speed_kmh', 'target_speed_kmh']
print(f'Found {len(pairs)} complete recording pairs.')
display(example_frame[original_columns].head(12))
unit_audit = pd.DataFrame({'interpretation':['header literally as km/h','recorded value as m/s, converted to km/h'], 'example_mae_against_can_kmh':[np.mean(np.abs(example_frame.gps_speed_file_value-example_frame.target_speed_kmh)), np.mean(np.abs(example_frame.gps_speed_kmh-example_frame.target_speed_kmh))]})
display(unit_audit)
raw_plot = example_frame.iloc[:min(600, len(example_frame))].copy()
raw_plot['elapsed_s'] = (raw_plot.timestamp_ns - raw_plot.timestamp_ns.iloc[0]) * 1e-9
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(raw_plot.elapsed_s, raw_plot[['acc_x', 'acc_y', 'acc_z']]); axes[0].set_ylabel('raw acceleration (m/s^2)'); axes[0].legend(['x', 'y', 'z'])
axes[1].plot(raw_plot.elapsed_s, raw_plot[['gyro_x', 'gyro_y', 'gyro_z']]); axes[1].set_ylabel('raw angular rate (rad/s)'); axes[1].legend(['x', 'y', 'z'])
axes[2].plot(raw_plot.elapsed_s, raw_plot.gps_speed_kmh, label='phone GPS'); axes[2].plot(raw_plot.elapsed_s, raw_plot.target_speed_kmh, label='CAN target', alpha=.8)
axes[2].set_ylabel('speed (km/h)'); axes[2].set_xlabel('elapsed time (s)'); axes[2].legend()
fig.suptitle(f'Original aligned data: {example_frame.journey_id.iloc[0]}'); fig.tight_layout(); plt.show()

## 2. Deterministic preprocessing and retention audit

Each row passes through the package implementations of unit normalization, synchronization, orientation estimation, phone-to-vehicle calibration, gravity removal, quality checks, and fixed-rate resampling. This dataset is already SI and its paired channels share timestamps, so normalization and synchronization validate rather than visibly filter it. Rotation into Forward-Left-Up and gravity removal create the meaningful representation change.

The cache contains deterministic outputs only. Rebuild it after changing sensor-pipeline code or preprocessing settings.

In [ ]:
PREPROCESSING_CACHE = ARTIFACT_DIR / 'processed_journeys.joblib'
cache_key = {'preprocessing_revision': 4, 'sample_period_ns': CONFIG.sample_period_ns, 'minimum_calibration_confidence': CONFIG.minimum_calibration_confidence, 'calibration_history_s': CONFIG.calibration_history_s}
cached = joblib.load(PREPROCESSING_CACHE) if PREPROCESSING_CACHE.exists() and not REBUILD_PREPROCESSING else {}
cache_is_usable = cached.get('cache_key') == cache_key
if cache_is_usable:
    processed_journeys, skipped_journeys = cached['journeys'], cached['skipped']
    print(f'Loaded {len(processed_journeys)} cached journeys.')
else:
    processed_journeys, skipped_journeys = [], []
    for pair in pairs:
        try:
            aligned = read_aligned_journey(*pair, CONFIG)
            journey = replay_clean_journey(aligned, CONFIG)
            processed_journeys.append(journey)
            print(f'{pair[0]}: {len(journey.clean_imu):,} cleaned rows')
        except (KeyError, ValueError) as error:
            skipped_journeys.append({'journey_id': pair[0], 'reason': str(error)})
            print(f'{pair[0]}: skipped - {error}')
    joblib.dump({'cache_key': cache_key, 'journeys': processed_journeys, 'skipped': skipped_journeys}, PREPROCESSING_CACHE)
if len(processed_journeys) < CONFIG.cv_folds + 1:
    raise RuntimeError('Too few usable journeys for grouped CV and a frozen test set.')

In [ ]:
audit_frame = pd.DataFrame([journey.audit for journey in processed_journeys]).fillna(0)
skipped_frame = pd.DataFrame(skipped_journeys)
display(audit_frame.sort_values('resampled_rows', ascending=False))
if not skipped_frame.empty: display(skipped_frame)
stages = ['source_rows', 'normalized_and_synchronized', 'rows_before_calibration', 'rows_using_held_calibration', 'rows_below_calibration_confidence', 'quality_rejected_rows', 'quality_accepted_rows', 'resampled_rows']
source_total = max(1, int(audit_frame.source_rows.sum()))
retention = pd.DataFrame({'stage_or_reason': stages, 'rows': [int(audit_frame.get(stage, pd.Series(dtype=float)).sum()) for stage in stages]})
retention['percent_of_aligned_input'] = retention.rows / source_total * 100
display(retention)

## 3. Original versus preprocessed data

The table uses matching resampled timestamps. Raw is the arbitrary phone frame. Vehicle specific force is after mounting calibration but before gravity removal. Clean is Forward-Left-Up linear acceleration after gravity compensation plus the rotated gyroscope. Plot downsampling affects display only, never training.

In [ ]:
visual_journey = max(processed_journeys, key=lambda item: len(item.clean_imu))
comparison_table = pd.DataFrame({'timestamp_ns': visual_journey.timestamps_ns, **{name: visual_journey.raw_imu[:, i] for i, name in enumerate(RAW_FEATURE_NAMES)}, **{f'vehicle_before_gravity_{name}': visual_journey.vehicle_imu_before_gravity_removal[:, i] for i, name in enumerate(('ax','ay','az','gx','gy','gz'))}, **{f'clean_{name}': visual_journey.clean_imu[:, i] for i, name in enumerate(VELOCITY_MODEL_FEATURE_NAMES)}, 'gps_speed_kmh': visual_journey.gps_speed_mps * 3.6, 'can_target_kmh': visual_journey.target_speed_mps * 3.6, 'calibration_confidence': visual_journey.calibration_confidence})
display(comparison_table.head(12))
n = min(600, len(visual_journey.clean_imu)); elapsed = (visual_journey.timestamps_ns[:n] - visual_journey.timestamps_ns[0]) * 1e-9
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
axes[0].plot(elapsed, visual_journey.raw_imu[:n, :3]); axes[0].set_ylabel('raw phone accel'); axes[0].legend(['x','y','z'])
axes[1].plot(elapsed, visual_journey.vehicle_imu_before_gravity_removal[:n, :3]); axes[1].set_ylabel('vehicle specific force'); axes[1].legend(['forward','left','up'])
axes[2].plot(elapsed, visual_journey.clean_imu[:n, :3]); axes[2].set_ylabel('clean linear accel'); axes[2].legend(['forward','left','up'])
axes[3].plot(elapsed, visual_journey.gps_speed_mps[:n]*3.6, label='GPS'); axes[3].plot(elapsed, visual_journey.target_speed_mps[:n]*3.6, label='CAN'); axes[3].plot(elapsed, visual_journey.calibration_confidence[:n]*100, label='confidence x100')
axes[3].set_xlabel('elapsed time (s)'); axes[3].set_ylabel('speed / confidence'); axes[3].legend(); fig.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for channel, axis in enumerate(axes.flat):
    axis.hist(visual_journey.raw_imu[:, channel], bins=60, density=True, alpha=.45, label='raw')
    axis.hist(visual_journey.clean_imu[:, channel], bins=60, density=True, alpha=.45, label='clean')
    axis.set_title(VELOCITY_MODEL_FEATURE_NAMES[channel]); axis.legend()
fig.suptitle('Original and cleaned distributions'); fig.tight_layout(); plt.show()

## 4. Physics preflight, then simulate GNSS blackouts

Before model training, the next cell checks two basic invariants: quiet vehicle-frame samples should become near-zero after gravity removal, and five-second integrated forward acceleration should roughly agree with five-second CAN speed change. If this preflight fails, inspect orientation, calibration, and gravity removal before spending hours on full hyperparameter search.

Once it passes, each example begins at a saved GNSS anchor; all later GNSS is hidden. Forward acceleration is integrated to 5, 10, 20, 30, 60, 90, and 120 seconds as a context feature and deterministic baseline. The learned target is the actual speed change since the anchor. Episodes never cross a journey, sensor gap, or quality reset.

In [ ]:
physics_audit = preprocessing_physics_audit(processed_journeys, CONFIG)
display(physics_audit)
if SEARCH_PROFILE == 'full' and not bool(physics_audit.preflight_pass.all()):
    warning = ('Physics preflight failed: this full run is diagnostic only, '
               'not a production-model selection. Fix calibration before trusting its winner.')
    if not ALLOW_DIAGNOSTIC_FULL_RUN:
        raise RuntimeError(warning + ' Set ALLOW_DIAGNOSTIC_FULL_RUN=True to continue deliberately.')
    print('WARNING:', warning)
dataset = build_blackout_dataset(processed_journeys, CONFIG)
display(pd.DataFrame({'examples':[len(dataset)], 'journeys':[len(np.unique(dataset.journey_ids))], 'clean_shape':[str(dataset.clean_windows.shape)], 'raw_shape':[str(dataset.raw_windows.shape)], 'context':[', '.join(CONTEXT_FEATURE_NAMES)]}))
display(pd.Series(dataset.horizons_s).value_counts().sort_index().rename_axis('horizon_s').to_frame('examples'))
print('Post-anchor GNSS is not stored in any model input tensor.')

## 5. Freeze test journeys and create shared grouped folds

A deterministic grouped search chooses a test set representative in size, mean speed, high-speed share, and horizon mix. A journey-level allocator then balances five development folds by example count, speed categories, and horizon coverage while keeping every journey indivisible. RF, CNN, GRU, and the historical controls all reuse these exact indices.

In [ ]:
development_index, test_index = choose_grouped_holdout(dataset, CONFIG)
development_groups, test_groups = set(dataset.journey_ids[development_index]), set(dataset.journey_ids[test_index])
assert development_groups.isdisjoint(test_groups)
cv_folds = make_grouped_cv_folds(dataset, development_index, CONFIG)
assert_group_isolation(dataset, cv_folds)
fold_audit = fold_audit_frame(dataset, development_index, test_index, cv_folds)
display(fold_audit)
print(f'Development={len(development_index):,}; frozen test={len(test_index):,}. Leakage checks passed.')

## 6. Features, grids, and grouped model selection

RF receives 52 IMU descriptors: mean, standard deviation, min, max, RMS, IQR, demeaned zero-crossing rate, and non-DC 0–3 Hz energy ratio per channel, plus mean/std of acceleration and gyro magnitudes. Five context values make 57 total. Anchor-delta neural models use weighted Huber loss in m/s. GRU search includes unidirectional and bidirectional fixed-window models. Bidirectionality is still causal because the entire window precedes the prediction time, but a selected bidirectional model must reprocess the window instead of carrying a simple streaming state. The CNN pads only its observed past.

The next cell is expensive in full mode. Candidate ranking uses mean macro-journey validation MAE; standard deviation exposes unstable candidates. The frozen test journeys remain untouched.

In [ ]:
rf_check = rf_imu_features(dataset.clean_windows[:min(64, len(dataset))], CONFIG.sample_period_s)
assert rf_check.shape[1] == 52 and np.isfinite(rf_check).all()
RAW_GRU_PARAMETERS = {'hidden_size':64, 'num_layers':2, 'bidirectional':False, 'dropout':.2, 'learning_rate':1e-3}
CLEAN_CNN_PARAMETERS = {'conv_blocks':3, 'filter_base':32, 'kernel_size':3, 'dropout':.2, 'learning_rate':1e-3}
raw_gru_cv = tune_absolute_neural_baseline(label='raw_absolute_gru', family='gru', sequence=dataset.raw_windows, parameters=RAW_GRU_PARAMETERS, dataset=dataset, folds=cv_folds, config=CONFIG, device=DEVICE)
clean_cnn_cv = tune_absolute_neural_baseline(label='clean_absolute_cnn', family='cnn', sequence=dataset.clean_windows, parameters=CLEAN_CNN_PARAMETERS, dataset=dataset, folds=cv_folds, config=CONFIG, device=DEVICE)
anchor_delta_gru_cv = tune_neural_family('gru', dataset, cv_folds, CONFIG, DEVICE)
anchor_delta_cnn_cv = tune_neural_family('cnn', dataset, cv_folds, CONFIG, DEVICE)
anchor_delta_rf_cv = tune_random_forest(dataset, cv_folds, CONFIG)
selection_results = pd.concat([raw_gru_cv, clean_cnn_cv, anchor_delta_gru_cv, anchor_delta_cnn_cv, anchor_delta_rf_cv], ignore_index=True).sort_values('cv_macro_mae_mps').reset_index(drop=True)
selection_results['cv_macro_mae_kmh'] = selection_results.cv_macro_mae_mps * 3.6
selection_results['cv_macro_std_kmh'] = selection_results.cv_macro_mae_std_mps * 3.6
display(selection_results)

## 7. Final refit and one frozen-test evaluation

Neural winners are refitted on all development journeys for the median best epoch found in CV. Constant-anchor and pure-integration baselines need no fitting. The test set is opened once, after all choices are fixed.

In [ ]:
def winner(model_name): return selection_results.loc[selection_results.model == model_name].iloc[0]
predictions = {'constant_anchor':np.maximum(0, dataset.context[test_index,0]), 'pure_integration':np.maximum(0, dataset.context[test_index,1])}
trained_models, trained_scalers = {}, {}
for label, family, sequence, parameters in [('raw_absolute_gru','gru',dataset.raw_windows,RAW_GRU_PARAMETERS), ('clean_absolute_cnn','cnn',dataset.clean_windows,CLEAN_CNN_PARAMETERS)]:
    row = winner(label); epochs = max(1, int(round(np.median(row.fold_best_epochs))))
    model, scalers = fit_final_absolute_neural(family=family, sequence=sequence, parameters=parameters, epochs=epochs, dataset=dataset, development_index=development_index, config=CONFIG, device=DEVICE)
    trained_models[label], trained_scalers[label] = model, scalers
    predictions[label] = predict_final_absolute_neural(model, scalers, sequence, test_index, CONFIG, DEVICE)
for family in ('gru','cnn'):
    label = f'anchor_delta_{family}'; row = winner(label); epochs = max(1, int(round(np.median(row.fold_best_epochs))))
    model, scalers = fit_final_neural(family, row.parameters, epochs, dataset, development_index, CONFIG, DEVICE)
    trained_models[label], trained_scalers[label] = model, scalers
    predictions[label] = predict_final_neural(model, scalers, dataset, test_index, CONFIG, DEVICE)
rf_row = winner('anchor_delta_random_forest')
trained_models['anchor_delta_random_forest'] = fit_final_forest(rf_row.parameters, dataset, development_index, CONFIG)
predictions['anchor_delta_random_forest'] = predict_final_forest(trained_models['anchor_delta_random_forest'], dataset, test_index, CONFIG)
comparison = pd.DataFrame([{'model':name, **regression_metrics(dataset.target_speed_mps[test_index], prediction, dataset.journey_ids[test_index])} for name, prediction in predictions.items()]).sort_values('macro_journey_mae_mps').reset_index(drop=True)
display(comparison)

In [ ]:
error_detail = detailed_error_frame(dataset, test_index, predictions)
horizon_errors = error_detail[error_detail.slice == 'horizon']
fig, axes = plt.subplots(1, 2, figsize=(15,5))
for name, rows in horizon_errors.groupby('model'): axes[0].plot(rows.value, rows.macro_journey_mae_kmh, marker='o', label=name)
axes[0].set(xlabel='seconds since GNSS anchor', ylabel='macro journey MAE (km/h)', title='Error growth during GNSS loss'); axes[0].legend(fontsize=8)
axes[1].bar(comparison.model, comparison.macro_journey_mae_kmh); axes[1].set(ylabel='macro journey MAE (km/h)', title='Frozen-test comparison'); axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout(); plt.show()
best_model = comparison.iloc[0].model; best_horizon = horizon_errors[horizon_errors.model == best_model].set_index('value')
targets = {30:5., 60:8., 120:12.}
acceptance = pd.DataFrame([{'horizon_s':h, 'target_mae_kmh':target, 'observed_mae_kmh':float(best_horizon.loc[h,'macro_journey_mae_kmh']), 'passed':bool(best_horizon.loc[h,'macro_journey_mae_kmh'] <= target)} for h, target in targets.items() if h in best_horizon.index])
display(acceptance)

## 8. Latency and reproducible artifacts

Timings use batch size one after warm-up and report median and p95. Model-only isolates neural execution. Feature-plus-model includes scaling and residual composition. A separate full-journey replay estimates amortized deterministic preprocessing per incoming sample; actual phone hardware must still be benchmarked.

In [ ]:
latency_rows = []; sample_index = int(test_index[0])
for label in ('anchor_delta_gru','anchor_delta_cnn'):
    model, scalers = trained_models[label], trained_scalers[label]
    xs = torch.from_numpy(transform_sequence(scalers.sequence, dataset.clean_windows[[sample_index]])).to(DEVICE)
    cs = torch.from_numpy(transform_context(scalers.context, dataset.context[[sample_index]])).to(DEVICE); model.eval()
    latency_rows.append({'model':label, 'stage':'model_only', **measure_latency_ms(lambda model=model: model(xs,cs), CONFIG, DEVICE)})
    latency_rows.append({'model':label, 'stage':'feature_plus_model', **measure_latency_ms(lambda label=label: predict_final_neural(trained_models[label], trained_scalers[label], dataset, np.asarray([sample_index]), CONFIG, DEVICE), CONFIG, DEVICE)})
latency_rows.append({'model':'anchor_delta_random_forest', 'stage':'feature_plus_model', **measure_latency_ms(lambda: predict_final_forest(trained_models['anchor_delta_random_forest'], dataset, np.asarray([sample_index]), CONFIG), CONFIG)})
latency_frame = pd.DataFrame(latency_rows); display(latency_frame)

In [ ]:
selection_results.to_csv(ARTIFACT_DIR/'cv_selection.csv', index=False); comparison.to_csv(ARTIFACT_DIR/'frozen_test_comparison.csv', index=False)
error_detail.to_csv(ARTIFACT_DIR/'error_by_horizon_and_speed.csv', index=False); fold_audit.to_csv(ARTIFACT_DIR/'fold_audit.csv', index=False)
audit_frame.to_csv(ARTIFACT_DIR/'preprocessing_audit.csv', index=False); skipped_frame.to_csv(ARTIFACT_DIR/'skipped_journeys.csv', index=False)
latency_frame.to_csv(ARTIFACT_DIR/'latency.csv', index=False); acceptance.to_csv(ARTIFACT_DIR/'acceptance_targets.csv', index=False)
prediction_frame = pd.DataFrame({'journey_id':dataset.journey_ids[test_index], 'anchor_timestamp_ns':dataset.anchor_timestamps_ns[test_index], 'end_timestamp_ns':dataset.end_timestamps_ns[test_index], 'horizon_s':dataset.horizons_s[test_index], 'anchor_speed_mps':dataset.context[test_index,0], 'integrated_speed_mps':dataset.context[test_index,1], 'actual_speed_mps':dataset.target_speed_mps[test_index], **{f'{name}_prediction_mps':value for name,value in predictions.items()}})
prediction_frame.to_csv(ARTIFACT_DIR/'frozen_test_predictions.csv', index=False)
for label in ('raw_absolute_gru','clean_absolute_cnn','anchor_delta_gru','anchor_delta_cnn'):
    torch.save({'model_name':label, 'state_dict':trained_models[label].state_dict(), 'sequence_scaler':trained_scalers[label].sequence, 'context_scaler':trained_scalers[label].context, 'feature_names':list(VELOCITY_MODEL_FEATURE_NAMES), 'context_names':list(CONTEXT_FEATURE_NAMES) if label.startswith('anchor_delta_') else [], 'target_unit':'m/s'}, ARTIFACT_DIR/f'{label}.pt')
joblib.dump(trained_models['anchor_delta_random_forest'], ARTIFACT_DIR/'anchor_delta_random_forest.joblib')
manifest = {'config':config_dict(CONFIG), 'development_journeys':sorted(development_groups), 'test_journeys':sorted(test_groups), 'feature_names':list(VELOCITY_MODEL_FEATURE_NAMES), 'context_names':list(CONTEXT_FEATURE_NAMES), 'rf_imu_feature_count':52, 'rf_total_feature_count':57, 'target_unit':'m/s', 'selection_metric':'five-fold macro journey MAE in m/s', 'gnss_policy':'only blackout-start speed is exposed'}
(ARTIFACT_DIR/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Saved experiment bundle to {ARTIFACT_DIR}')

## Interpretation

Choose the candidate using development CV before reading frozen-test results. The anchored model must beat both constant-speed hold and pure integration; merely beating the old absolute-speed GRU is insufficient. Judge the 30, 60, and 120 second macro-journey targets separately. If a gate fails, inspect journey, speed-bin, calibration, and horizon diagnostics before widening the neural search.

This notebook does not train uncertainty. Once a velocity model passes accuracy and latency checks, its predicted speed, anchor age, model identity, calibration confidence, and sensor-quality context become inputs to the separately evaluated uncertainty engine.